# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library in the context of the [FAIR^2 dataset package](https://doi.org/10.71728/senscience.vq4a-28xa).

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Note: dataset.metadata is an object, attributes accessed via .<field>
print(f"Dataset Name: {dataset.metadata.name}")
print(f"Dataset Description: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record set @ids and summarize their fields and columns
print("Available Record Sets (by @id):")
for rs in dataset.record_sets:
    print(f"- RecordSet: {rs.id}")
    # List fields (by @id and label)
    if hasattr(rs, 'fields') and rs.fields:
        print("  Fields:")
        for f in rs.fields:
            name = f.label if hasattr(f, 'label') and f.label else f.id
            print(f"    - {f.id}: {name} (dataType: {getattr(f, 'data_type', None)})")
    # List columns (by @id and label), if present
    if hasattr(rs, 'columns') and rs.columns:
        print("  Columns:")
        for c in rs.columns:
            cname = c.label if hasattr(c, 'label') and c.label else c.id
            print(f"    - {c.id}: {cname}")
    print()

# Show record sets info as a list for reference
record_set_ids = [rs.id for rs in dataset.record_sets]
print("Record Set @ids:", record_set_ids)

## 3. Data Extraction
Load data from record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# For demonstration, we'll load all record sets into pandas DataFrames
dataframes = {}
for record_set_id in record_set_ids:
    try:
        records_iter = dataset.records(record_set=record_set_id)
        # Convert records generator to list and then DataFrame
        records = list(records_iter)
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} rows for RecordSet: {record_set_id}")
    except Exception as e:
        print(f"Could not load RecordSet {record_set_id}: {e}")

# For exploration, pick first available record set with tabular data:
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print(f"\nColumns in main record set ({main_record_set_id}):")
    print(dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes example operations: removing outliers, transforming numeric distributions, and grouping by categorical fields.

In [ ]:
# Choose a numeric field for analysis by inspecting columns
if main_record_set_id:
    main_df = dataframes[main_record_set_id]
    print("First few columns:", main_df.columns[:10].tolist())
    # Example: try typical numeric column names observed in proteomics
    possible_numeric_fields = ['cr:MW', 'cr:peptides', 'cr:coverage', 'cr:Abundance', 'MW', 'coverage_pct',
                               'Peptides', 'NormalizedAbundance', 'cr:MW_(Da)', 'cr:coverage_%', 'cr:MW', 'cr:pI']
    chosen_numeric = None
    for f in possible_numeric_fields:
        if f in main_df.columns:
            chosen_numeric = f
            break
    if chosen_numeric is None:
        numeric_cols = main_df.select_dtypes('number').columns
        if len(numeric_cols) > 0:
            chosen_numeric = numeric_cols[0]

    print(f"Numeric field chosen: {chosen_numeric}")

    # Choose a group field for demonstration (like a protein group or description)
    possible_group_fields = ['cr:description', 'cr:accession', 'Description', 'Protein']
    group_field = None
    for g in possible_group_fields:
        if g in main_df.columns:
            group_field = g
            break
    if not group_field and len(main_df.columns) > 1:
        # Fallback on the second column if present
        group_field = main_df.columns[1]
    print(f"Grouping field chosen: {group_field}\n")

    # Apply a filter on the numeric column
    threshold = main_df[chosen_numeric].quantile(0.75) if chosen_numeric and chosen_numeric in main_df.columns else None
    filtered_df = main_df
    if chosen_numeric and threshold is not None:
        filtered_df = main_df[main_df[chosen_numeric] > threshold]
        print(f"Filtered records with {chosen_numeric} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{chosen_numeric}_normalized"] = (
            (filtered_df[chosen_numeric] - filtered_df[chosen_numeric].mean()) / filtered_df[chosen_numeric].std()
        )
        print(f"\nNormalized '{chosen_numeric}' for filtered records:")
        print(filtered_df[[chosen_numeric, f"{chosen_numeric}_normalized"]].head())

    # Group by the chosen field and calculate the mean for numeric columns
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"\nGrouped data by {group_field}:")
        print(grouped_df.head())
else:
    print("No available record set to perform EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and chosen_numeric and chosen_numeric in main_df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(main_df[chosen_numeric].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {chosen_numeric}")
    plt.xlabel(chosen_numeric)
    plt.ylabel("Count")
    plt.show()

    if group_field and group_field in main_df.columns:
        plt.figure(figsize=(10, 6))
        # Show mean of numeric field per group, for up to 20 most frequent groups
        gmeans = main_df.groupby(group_field)[chosen_numeric].mean().sort_values(ascending=False)[:20]
        sns.barplot(x=gmeans.values, y=gmeans.index, orient='h')
        plt.title(f"Mean {chosen_numeric} by {group_field} (top 20)")
        plt.xlabel(f"Mean {chosen_numeric}")
        plt.ylabel(group_field)
        plt.show()

## 6. Conclusion
This notebook demonstrated loading, exploring, and visualizing the FAIR^2 protein dataset using `mlcroissant`. Using Croissant schema `@id` references, we explored record sets, fields, processed numerical data, and visualized protein measures. For deeper biological or machine learning insights, further curation and domain-specific analysis would follow.

You are encouraged to further explore the dataset with biological or computational research questions based on these workflows!